<a href="https://colab.research.google.com/github/Akshbisht/Akshbisht.github.io/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# RESEARCH-PAPER STYLE DR GRADING SYSTEM
# EfficientNetV2M + ConvNeXt-B ENSEMBLE
# EyePACS subset (online) + Messidor-2 (local)
# =========================================================

!pip install datasets tensorflow opencv-python matplotlib pandas -q

import os, cv2, numpy as np, tensorflow as tf, pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetV2M, ConvNeXtBase
from sklearn.model_selection import train_test_split



In [ ]:
IMG_SIZE = 384

# =========================================================
# 1️⃣ LOAD EYEPACS SUBSET (TRAINING)
# =========================================================

print("Loading EyePACS subset...")
ds = load_dataset("bumbledeep/eyepacs")["train"]

X_eye, y_eye = [], []

for item in ds:
    img = np.array(item["image"])
    X_eye.append(img)
    y_eye.append(item["label"])

X_eye = np.array(X_eye)
y_eye = np.array(y_eye)

# =========================================================
# 2️⃣ PAPER-STYLE PREPROCESSING
# =========================================================

def preprocess(img):
    h, w, _ = img.shape
    center = (w//2, h//2)
    r = min(center)

    mask = np.zeros((h, w), np.uint8)
    cv2.circle(mask, center, r, 255, -1)

    img = cv2.bitwise_and(img, img, mask=mask)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    blur = cv2.GaussianBlur(img, (0,0), sigmaX=10)
    img = cv2.addWeighted(img, 4, blur, -4, 128)

    return img / 255.0

X_eye = np.array([preprocess(i) for i in X_eye])

#

Loading EyePACS subset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:

# 3️⃣ LOAD MESSIDOR-2 (LOCAL TEST SET)
# =========================================================

messidor_path = "C:\\Users\\akash\\Downloads\\IMAGES"    # <<< CHANGE
pairs_csv = "C:\\Users\\akash\\Downloads\\messidor-2 (2).csv"         # <<< CHANGE

pairs_df = pd.read_csv(pairs_csv)

X_mes, y_mes = [], []

for cls in range(5):
    cls_dir = os.path.join(messidor_path, str(cls))
    if not os.path.exists(cls_dir):
        continue

    for img_name in os.listdir(cls_dir):
        img_path = os.path.join(cls_dir, img_name)
        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = preprocess(img)

        X_mes.append(img)
        y_mes.append(cls)

X_mes = np.array(X_mes)
y_mes = np.array(y_mes)

# =========================================================
# 4️⃣ TRAIN / VALIDATION SPLIT
# =========================================================

X_train, X_val_eye, y_train, y_val_eye = train_test_split(
    X_eye, y_eye,
    test_size=0.1,
    stratify=y_eye,
    random_state=42
)

X_val = np.concatenate([X_val_eye, X_mes])
y_val = np.concatenate([y_val_eye, y_mes])

# =========================================================
# 5️⃣ DATA AUGMENTATION
# =========================================================

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# =========================================================
# 6️⃣ ENSEMBLE MODEL (EffNetV2M + ConvNeXt-B)
# =========================================================

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = augment(inputs)

# EfficientNetV2 branch
effnet = EfficientNetV2M(include_top=False,
                         weights="imagenet",
                         input_tensor=x)
x1 = layers.GlobalAveragePooling2D()(effnet.output)

# ConvNeXt branch
convnext = ConvNeXtBase(include_top=False,
                        weights="imagenet",
                        input_tensor=x)
x2 = layers.GlobalAveragePooling2D()(convnext.output)

# Fusion
x = layers.Concatenate()([x1, x2])

x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(5, activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# =========================================================
# 7️⃣ TRAIN MODEL
# =========================================================

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=8
)

# =========================================================
# 8️⃣ QUALITY FACTOR
# =========================================================

def quality_factor(probs):
    s = np.sort(probs)[::-1]
    return s[0] / s[1]

# =========================================================
# 9️⃣ GRAD-CAM
# =========================================================

def grad_cam(model, img, layer_name="top_activation"):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img)
        loss = preds[:, tf.argmax(preds[0])]

    grads = tape.gradient(loss, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))

    conv_out = conv_out[0]
    heatmap = conv_out @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    return np.maximum(heatmap, 0) / tf.reduce_max(heatmap)

# =========================================================
# 🔟 TEST SAMPLE
# =========================================================

sample = X_val[0:1]

pred = model.predict(sample)

print("Predicted class:", np.argmax(pred))
print("Quality factor:", quality_factor(pred[0]))

heatmap = grad_cam(model, sample)

plt.imshow(heatmap, cmap="jet")
plt.axis("off")